In [10]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType


In [14]:
#inserts mutation amino acid in corresponding position in protein sequence
def insert_wt(seq, pos, wt_aa):
    seq_list = list(seq)
    pos = int(pos)
    if pos < len(seq_list):
        seq_list[pos] = wt_aa
    return ''.join(seq_list)

In [ ]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [7]:
df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 602882 entries, 0 to 602881
Data columns (total 19 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   domain_ID                 602882 non-null  object 
 1   uniprot_ID                602882 non-null  object 
 2   aa_seq                    602882 non-null  object 
 3   wt_aa                     602360 non-null  object 
 4   position                  602360 non-null  float64
 5   mut_aa                    602360 non-null  object 
 6   STOP                      575258 non-null  object 
 7   input_count_rep1          575258 non-null  float64
 8   input_count_rep2          575258 non-null  float64
 9   input_count_rep3          575258 non-null  float64
 10  output_count_rep1         575258 non-null  float64
 11  output_count_rep2         575258 non-null  float64
 12  output_count_rep3         575258 non-null  float64
 13  mean_input_count          602882 non-null  f

Get data for one domain.  In this case P07316_PF00030_87.  Translate position by initial domain position.

In [ ]:
df_one_protein = df.where(df['domain_ID'] == 'P07316_PF00030_87').dropna()

dom_position = df_one_protein['position'] - 87.0
df_one_protein.insert(loc=0, column='real_position', value=dom_position)
df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

df_one_protein_ns.head()



,real_position,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
121785,0.0,P07316_PF00030_87,P07316,AAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,A,False,645.0,476.0,327.0,202.0,183.0,264.0,482.6667,0.067864,0.006483,0.163960,0.070917,385.0
121786,0.0,P07316_PF00030_87,P07316,CAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,C,False,1155.0,733.0,502.0,198.0,257.0,578.0,796.6667,0.061549,0.005723,0.094874,0.062609,385.0
121787,0.0,P07316_PF00030_87,P07316,DAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,D,False,1079.0,784.0,566.0,197.0,220.0,399.0,809.6667,0.054534,0.006000,0.018137,0.065632,385.0
121788,0.0,P07316_PF00030_87,P07316,EAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,E,False,305.0,251.0,187.0,75.0,94.0,182.0,247.6667,0.067474,0.008372,0.159687,0.091586,385.0
121789,0.0,P07316_PF00030_87,P07316,FAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,F,False,803.0,628.0,463.0,241.0,412.0,729.0,631.3333,0.083318,0.004906,0.333017,0.053670,385.0


Insert mutation into correct position of protein sequence.

In [ ]:
df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
                                                      insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
                                                      axis=1)


In [18]:
df_mutation = df_one_protein_ns[['real_position','mut_aa','normalized_fitness']]
df_mutation

,real_position,mut_aa,normalized_fitness
121785,0.0,A,0.163960
121786,0.0,C,0.094874
121787,0.0,D,0.018137
121788,0.0,E,0.159687
121789,0.0,F,0.333017
...,...,...,...
123520,0.0,S,0.000670
123521,0.0,T,0.058108
123522,0.0,V,0.049949
123523,0.0,W,0.022983


Initializing base model Pro Gen 2

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = 'cpu'
print(f"Using {device} device")

model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2(model_name)

Initializing LoRA

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "key", "value", "output.dense"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)
model = get_peft_model(base_model, lora_config)

Set device and optimizer

In [ ]:
model.to(device)
optimizer = torch.optim.AdamW(model.parameters, lr=1e-5)